Fine-tuning RoBERTa-base on SST-2

Goal:fine-tune RoBERTa-base on SST-2, report accuracy and F1 on the validation set (872 sentences) and save the model to Google Drive.

Before running: Runtime → Change runtime type → T4 GPU.

In [ ]:
# check that a GPU is available
!nvidia-smi -L

Mount Google Drive
Colab deletes all files when the session ends, so the trained model is saved to Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# folder on Google Drive where the trained model will be saved
MODEL_DIR = '/content/drive/MyDrive/xai/roberta-sst2'

Clone the repository and install packages
The code is taken from GitHub, so Colab runs exactly the same code as the repository. 

In [ ]:
GITHUB_URL = 'https://github.com/alinamoldovan0305/xai-ig-shap-roberta'   

%cd /content
!rm -rf xai-ig-shap-roberta          # remove an old copy, if there is one
!git clone -q $GITHUB_URL            # download the code from GitHub
%cd /content/xai-ig-shap-roberta
!pip install -q -r requirements.txt  # install the packages from requirements.txt

Check the data
Loads SST-2 and EmoBank with `src/data.py` and prints their size.

In [ ]:
!python -m src.data

Test run :
Trains on only 2,000 sentences for one epoch to check that everything works.
The accuracy here is not meaningful.

In [ ]:
!python -m src.train_sst2 --max_train_samples 2000 --epochs 1 --output_dir /content/test-run

Full training
67,349 training sentences, 2 epochs, learning rate 2e-5, batch size 32, max length 128 tokens.
The model is saved to Google Drive.

In [ ]:
!python -m src.train_sst2 --output_dir $MODEL_DIR

Results
Accuracy and F1 on the 872 validation sentences.

In [ ]:
import json

with open('results/sst2_model_metrics.json') as f:
    m = json.load(f)

print(f"Accuracy: {m['accuracy']:.4f}   F1: {m['f1']:.4f}   (n={m['n_eval']})")
m

Sanity check
Loads the saved model from Google Drive and predicts a few new sentences.

In [ ]:
from transformers import pipeline

# load the trained model from Drive (device=0 = use the GPU)
clf = pipeline('text-classification', model=MODEL_DIR, device=0)

examples = [
    "A gorgeous, witty, seductive movie.",
    "The plot is dull and the acting is even worse.",
    "It's not a bad film, but it never quite takes off.",
    "The roller coaster was thrilling and terrifying.",
]
for text, pred in zip(examples, clf(examples)):
    print(f"{pred['label']:>8}  {pred['score']:.3f}  {text}")